# Election Bloc Change Prediction Project
## Notebook 00 — Historical redesign and reproducibility setup

The project now uses Knesset elections **16–25** instead of only 21–25. The main question remains:

> Do locality-level demographic and socioeconomic characteristics improve prediction of voting-bloc change beyond previous-election persistence?

### Design
- Four modeled blocs: `Right`, `Center_Left`, `Haredi`, `Arab`.
- Main target: change in CLR-transformed bloc composition.
- Primary baseline: repeat the previous election (`delta = 0`).
- Final untouched test: `K24_to_K25`.
- Development evaluation: expanding-window temporal backtests plus grouped unseen-locality validation.
- Historical demographic sources are aligned **as-of** the latest year strictly before the election year.

### Notebook sequence
00 setup → 01 elections and mappings → 02 transition panel → 03 EDA/baselines → 04 historical features → 05 temporal delta modeling → 06 unseen localities and candidate freeze.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, time, re, unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 300)
REPO_URL = "https://github.com/IfatDav/Election_Bloc_Prediction_Project.git"
DEFAULT_REPO = Path("/content/Election_Bloc_Prediction_Project")

def locate_repo():
    candidates=[]
    if os.getenv("ELECTION_PROJECT_ROOT"):
        candidates.append(Path(os.environ["ELECTION_PROJECT_ROOT"]).expanduser())
    cwd=Path.cwd().resolve()
    candidates += [cwd, *cwd.parents, DEFAULT_REPO]
    for p in candidates:
        if (p/"data"/"raw").exists(): return p
    if Path("/content").exists():
        if DEFAULT_REPO.exists(): shutil.rmtree(DEFAULT_REPO)
        subprocess.run(["git","clone","--depth","1",REPO_URL,str(DEFAULT_REPO)],check=True)
        return DEFAULT_REPO
    raise FileNotFoundError("Repository not found. Set ELECTION_PROJECT_ROOT.")

REPO_ROOT=locate_repo()
RAW_DIR=REPO_ROOT/"data"/"raw"
INTERIM_DIR=REPO_ROOT/"data"/"interim"
PROCESSED_DIR=REPO_ROOT/"data"/"processed"
REPORTS_DIR=REPO_ROOT/"reports"
TABLES_DIR=REPORTS_DIR/"tables"
FIGURES_DIR=REPORTS_DIR/"figures"
SUMMARIES_DIR=REPORTS_DIR/"summaries"
MODELS_DIR=REPO_ROOT/"models"
for d in [INTERIM_DIR,PROCESSED_DIR,TABLES_DIR,FIGURES_DIR,SUMMARIES_DIR,MODELS_DIR]: d.mkdir(parents=True,exist_ok=True)
print("Repository:",REPO_ROOT)

In [ ]:
PROJECT_CONFIG={
    "elections": list(range(16,26)),
    "modeled_blocs": ["Right","Center_Left","Haredi","Arab"],
    "final_test_transition": "K24_to_K25",
    "demographic_alignment": "latest source year strictly before current election year",
    "primary_metric": "unweighted mean absolute error across four bloc shares",
    "secondary_metrics": ["weighted_mae","median_row_mae","macro_r2"],
}
config_path=SUMMARIES_DIR/"project_config_historical.json"
config_path.write_text(json.dumps(PROJECT_CONFIG,ensure_ascii=False,indent=2),encoding="utf-8")
print(json.dumps(PROJECT_CONFIG,ensure_ascii=False,indent=2))

Run this notebook once in a fresh Colab runtime. It does not fit a model; it records the frozen design and creates project folders.